In [3]:
import os
import pandas as pd
import numpy as np

# =========================================================
# SETTINGS
# =========================================================
FILE_PATH = r"C:\IDEAL_Programming\processed\final\IDEAL_final_hourly_dataset.csv"
OUTPUT_DIR = r"C:\IDEAL_Programming\Cases\Tight_Cases"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Final selected seasonal reference days
FINAL_DATES = {
    "winter": "01-28",
    "spring": "04-09",
    "summer": "07-28",
    "autumn": "10-30"
}

# =========================================================
# TIGHT CASE DEFINITIONS
# =========================================================
CASES = {
    "case_1_flat_2_medium": {
        "hometype": "flat",
        "residents": 2,
        "floor_min": 80,
        "floor_max": 90,
        "ui_floor_area": 85
    },
    "case_2_flat_1_small": {
        "hometype": "flat",
        "residents": 1,
        "floor_min": 45,
        "floor_max": 55,
        "ui_floor_area": 55
    },
    "case_3_flat_3_medium": {
        "hometype": "flat",
        "residents": 3,
        "floor_min": 80,
        "floor_max": 90,
        "ui_floor_area": 85
    },
    "case_4_house_4_large": {
        "hometype": "house_or_bungalow",
        "residents": 4,
        "floor_min": 100,
        "floor_max": 140,
        "ui_floor_area": 110
    },
    "case_5_house_2_medium": {
        "hometype": "house_or_bungalow",
        "residents": 2,
        "floor_min": 80,
        "floor_max": 110,
        "ui_floor_area": 85
    },
    "case_6_house_3_large": {
        "hometype": "house_or_bungalow",
        "residents": 3,
        "floor_min": 100,
        "floor_max": 140,
        "ui_floor_area": 110
    },
}

# =========================================================
# LOAD DATA
# =========================================================
df = pd.read_csv(FILE_PATH, low_memory=False)

df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df = df.dropna(subset=["timestamp"]).copy()

df["date"] = df["timestamp"].dt.date
df["hour"] = df["timestamp"].dt.hour
df["month_day"] = df["timestamp"].dt.strftime("%m-%d")
df["year"] = df["timestamp"].dt.year

# one row per home for metadata filtering
homes = df[[
    "home_id",
    "residents",
    "hometype",
    "total_floor_area_m2"
]].drop_duplicates(subset="home_id").copy()

# =========================================================
# HELPERS
# =========================================================
def get_case_homes(homes_df, case_def):
    subset = homes_df[
        (homes_df["hometype"] == case_def["hometype"]) &
        (homes_df["residents"] == case_def["residents"]) &
        (homes_df["total_floor_area_m2"] >= case_def["floor_min"]) &
        (homes_df["total_floor_area_m2"] <= case_def["floor_max"])
    ].copy()
    return subset


def get_real_profile_for_case_and_monthday(full_df, case_homes_df, target_month_day):
    temp = full_df[
        (full_df["home_id"].isin(case_homes_df["home_id"])) &
        (full_df["month_day"] == target_month_day)
    ].copy()

    if temp.empty:
        return None, 0, pd.DataFrame()

    counts = (
        temp.groupby(["home_id", "date"])["hour"]
        .nunique()
        .reset_index(name="n_hours")
    )

    valid_pairs = counts[counts["n_hours"] == 24][["home_id", "date"]].copy()

    if valid_pairs.empty:
        return None, 0, pd.DataFrame()

    temp = temp.merge(valid_pairs, on=["home_id", "date"], how="inner")

    real_profile = (
        temp.groupby("hour")["consumption_Wh"]
        .mean()
        .reindex(range(24))
    )

    n_valid_home_date_pairs = valid_pairs.shape[0]

    return real_profile, n_valid_home_date_pairs, valid_pairs


# =========================================================
# MAIN
# =========================================================
summary_rows = []
profile_rows = []

for case_name, case_def in CASES.items():
    case_homes = get_case_homes(homes, case_def)

    print("=" * 90)
    print(f"{case_name}")
    print(f"Metadata-matched homes: {len(case_homes)}")

    if case_homes.empty:
        print("No homes found for this case.")
        continue

    print(case_homes[["home_id", "residents", "hometype", "total_floor_area_m2"]].head())

    for season_name, md in FINAL_DATES.items():
        real_profile, n_valid_pairs, valid_pairs = get_real_profile_for_case_and_monthday(
            df, case_homes, md
        )

        years_used = []
        unique_homes_used = 0

        if valid_pairs is not None and not valid_pairs.empty:
            temp_dates = pd.to_datetime(valid_pairs["date"], errors="coerce")
            years_used = sorted(temp_dates.dt.year.dropna().unique().tolist())
            unique_homes_used = valid_pairs["home_id"].nunique()

        summary_rows.append({
            "case": case_name,
            "season": season_name,
            "month_day": md,
            "matched_homes_metadata": len(case_homes),
            "valid_home_date_pairs": n_valid_pairs,
            "unique_homes_used": unique_homes_used,
            "years_used": ", ".join(map(str, years_used)) if years_used else "",
            "floor_min": case_def["floor_min"],
            "floor_max": case_def["floor_max"]
        })

        print(f"\nSeason: {season_name} | Month-Day: {md}")

        if real_profile is None:
            print("No valid full 24h data found.")
        else:
            print(f"Valid home-date pairs: {n_valid_pairs}")
            print(f"Unique homes used: {unique_homes_used}")
            print(f"Years used: {years_used}")
            print(real_profile)

            for hour, value in real_profile.items():
                profile_rows.append({
                    "case": case_name,
                    "season": season_name,
                    "month_day": md,
                    "hour": int(hour),
                    "real_consumption_Wh": float(value),
                    "matched_homes_metadata": len(case_homes),
                    "valid_home_date_pairs": n_valid_pairs,
                    "unique_homes_used": unique_homes_used,
                    "years_used": ", ".join(map(str, years_used)) if years_used else "",
                    "floor_min": case_def["floor_min"],
                    "floor_max": case_def["floor_max"]
                })

# =========================================================
# SAVE OUTPUTS
# =========================================================
summary_df = pd.DataFrame(summary_rows)
profiles_df = pd.DataFrame(profile_rows)

summary_path = os.path.join(OUTPUT_DIR, "tight_case_season_summary.csv")
profiles_path = os.path.join(OUTPUT_DIR, "tight_case_season_real_profiles.csv")

summary_df.to_csv(summary_path, index=False)
profiles_df.to_csv(profiles_path, index=False)

print("\n" + "=" * 90)
print("FILES SAVED")
print("=" * 90)
print(summary_path)
print(profiles_path)

print("\nSUMMARY TABLE:")
print(summary_df)

case_1_flat_2_medium
Metadata-matched homes: 8
         home_id  residents hometype  total_floor_area_m2
120351   home118          2     flat                 83.0
321246   home150          2     flat                 86.5
941837   home267          2     flat                 82.5
1029792  home292          2     flat                 84.5
1070900  home310          2     flat                 81.5

Season: winter | Month-Day: 01-28
Valid home-date pairs: 7
Unique homes used: 5
Years used: [2017, 2018]
hour
0     155.714286
1      88.857143
2     131.571429
3      83.714286
4      80.857143
5     114.000000
6     118.714286
7     116.000000
8     196.857143
9     328.714286
10    305.285714
11    159.285714
12    230.285714
13    259.285714
14    181.571429
15    304.142857
16    277.714286
17    458.428571
18    823.142857
19    296.000000
20    283.428571
21    236.285714
22    178.714286
23    136.714286
Name: consumption_Wh, dtype: float64

Season: spring | Month-Day: 04-09
Valid home-dat